<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.1-rag-fundamentals/notebooks/exercises-6.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.1 — RAG Fundamentals
Netsetos GenAI Course v2.0 | Module 6

Document processing, embeddings, vector DBs, retrieval, generation, evaluation


In [ ]:
# pip install langchain langchain-openai chromadb pymupdf4llm rank-bm25 -q
import os
print('Ready for RAG fundamentals')


## Ex 1: Basic RAG Pipeline


In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 1. Documents
docs = [
    'RAG (Retrieval-Augmented Generation) grounds LLM responses in external documents.',
    'Chunking splits documents into retrievable pieces. 512 tokens is optimal.',
    'Hybrid search combines dense (embeddings) and sparse (BM25) retrieval.',
    'Reranking with cross-encoders improves precision by 10-25%.',
    'RAGAS evaluates RAG with faithfulness, relevancy, context metrics.',
]

# 2. Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.create_documents(docs)
print(f'{len(chunks)} chunks created')

# 3. Embed + Store
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
db = Chroma.from_documents(chunks, embeddings)

# 4. Retrieve
results = db.similarity_search('What is hybrid search?', k=3)
for i, r in enumerate(results):
    print(f'  [{i+1}] {r.page_content[:80]}...')

# 5. Generate
llm = ChatOpenAI(model='gpt-4o-mini')
context = '\n'.join([r.page_content for r in results])
prompt = f'Answer using ONLY this context:\n{context}\n\nQuestion: What is hybrid search?'
response = llm.invoke(prompt)
print(f'\nAnswer: {response.content}')


## Ex 2: PDF → Markdown → Chunk


In [ ]:
# pip install pymupdf4llm -q
# import pymupdf4llm
# md_text = pymupdf4llm.to_markdown('document.pdf')
# print(f'Extracted {len(md_text)} chars of Markdown')

# from langchain.text_splitter import MarkdownTextSplitter
# splitter = MarkdownTextSplitter(chunk_size=512, chunk_overlap=50)
# chunks = splitter.create_documents([md_text])
# print(f'{len(chunks)} chunks')

print('Best practice: PDF → pymupdf4llm → Markdown → Chunk')
print('pymupdf4llm preserves tables as Markdown tables')
print('MarkdownTextSplitter respects heading boundaries')


## Ex 3: Chunk Size Experiment


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text = 'A ' * 5000  # Simulated document
sizes = [128, 256, 512, 1024, 2048]
for size in sizes:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=int(size*0.1))
    chunks = splitter.create_documents([text])
    print(f'  {size:5d} tokens → {len(chunks):3d} chunks, overlap={int(size*0.1)}')

print('\nVecta Benchmark Results:')
print('  512 tokens: 69% accuracy (#1)')
print('  256 tokens: 65% accuracy')
print('  1024 tokens: 58% accuracy')
print('  Semantic: 54% (too fragmented)')


## Ex 4: Hybrid Search (Dense + BM25)


In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

# Documents
corpus = [
    'Error code ERR_CONNECTION_REFUSED on port 8080',
    'The server connection was refused when trying to connect',
    'Python pip install fails with SSL certificate error',
    'Machine learning models require GPU for training',
]

# BM25 sparse retrieval
tokenized = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized)
query = 'ERR_CONNECTION_REFUSED'
bm25_scores = bm25.get_scores(query.lower().split())
print('BM25 scores:', {corpus[i][:40]: f'{s:.2f}' for i, s in enumerate(bm25_scores)})

# RRF fusion (with dense would go here)
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

bm25_ranking = list(np.argsort(-bm25_scores))
print(f'\nBM25 catches exact error codes that embeddings miss!')
print(f'RRF fusion: score(d) = Σ 1/(60 + rank_i(d))')


## Ex 5: Embedding Model Comparison


In [ ]:
print('MTEB Embedding Rankings (2025-2026):')
models = [
    ('Qwen3-Embedding-8B', 70.58, 'Free', 'Best overall (open-source)'),
    ('Gemini Embedding 001', 68.32, '$0.15/M', 'Best retrieval (commercial)'),
    ('Voyage 3.5', '~67', '$0.06/M', 'Domain models'),
    ('Cohere Embed v4', '~67', '$0.12/M', 'Multimodal (text+image)'),
    ('BGE-M3', '~65', 'Free', 'Dense+sparse+multi-vector'),
    ('OpenAI 3-small', 62.3, '$0.02/M', 'Cheapest commercial'),
]
for name, score, cost, feat in models:
    print(f'  {name:25s} | MTEB: {str(score):6s} | {cost:8s} | {feat}')

print('\nMatryoshka: 256 dims outperforms ada-002 at 1536 dims')
print('Critical: switching models = re-embed entire corpus')


## Ex 6: Vector Database Selection


In [ ]:
print('Vector Database Decision Framework:')
dbs = [
    ('Chroma', 'Prototype', '<10M vectors', 'Free'),
    ('Qdrant', 'Production RAG', '<50M vectors', 'Free 1GB'),
    ('Weaviate', 'Hybrid search', '100M+', 'Free sandbox'),
    ('Milvus/Zilliz', 'Billion-scale', 'Billions', 'Free 5GB'),
    ('Pinecone', 'Zero ops', '100M+', 'Usage-based'),
    ('pgvector', 'PostgreSQL users', '<5M', 'Free'),
    ('LanceDB', 'Serverless/edge', '100M+', 'Free OSS'),
]
for name, best, scale, price in dbs:
    print(f'  {name:15s} | {best:20s} | {scale:12s} | {price}')

print('\nIndexing: HNSW (<100M) vs IVF-PQ (billions) vs ScaNN (Google)')


## Ex 7: RAG Prompt Template


In [ ]:
template = '''You are a technical support assistant.
Answer using ONLY the provided context.
If the context doesn't contain the answer, say "I don't have enough information."
Cite sources for every claim using [Source N] format.

<context>
[Source 1: config_guide.pdf, page 12]
{chunk_1}

[Source 2: api_docs.md, section 3.2]
{chunk_2}

[Source 3: faq.txt]
{chunk_3}
</context>

Question: {question}

Example format:
The maximum file size is 10MB [Source 1]. For larger files, use the batch API [Source 2].'''

print(template)
print('\n4 layers: role + grounding + numbered context + query + examples')
print('Lost-in-the-middle: place key docs at START and END, not middle')


## Ex 8: India RAG Challenges


In [ ]:
print('India RAG: Three Unique Challenges')
print('='*50)

print('\n1. Legacy Fonts (Kruti Dev, Shivaji01):')
print('   Devanagari mapped to Latin codepoints → gibberish')
print('   Fix: Font2Unicode conversion before text extraction')

print('\n2. Token Fertility:')
print('   Standard tokenizer: 4-8 tokens per Hindi word')
print('   Sarvam tokenizer:   1.4-2.1 tokens (2-4× savings)')
import re
hindi = 'मशीन लर्निंग एक कृत्रिम बुद्धिमत्ता'
print(f'   Hindi text: {hindi}')
print(f'   Approx GPT tokens: ~{len(hindi.split())*6}')
print(f'   Approx Sarvam tokens: ~{len(hindi.split())*2}')

print('\n3. Cross-Script Retrieval:')
print('   Hindi: highest scores')
print('   Hindi-Marathi (same script): 27.9% R@1')
print('   Hindi-Tamil (cross-script): 21.2% R@1')
print('   Solution: BGE-M3 + dual retrieval (Hindi dense + English BM25 + RRF)')


---
## Recap
RAG pipeline: Load → Chunk → Embed → Index → Retrieve → Generate. Best defaults: pymupdf4llm for PDFs, RecursiveCharacterTextSplitter(512, 50), Gemini/BGE-M3 for embeddings, Qdrant for production, hybrid search (+26% NDCG), reranking (+10-25%), grounded prompts with citations. RAG vs fine-tuning: RAG for knowledge, fine-tuning for behavior. Advanced patterns: Self-RAG, CRAG, GraphRAG, Agentic RAG — use the simplest that meets quality bar. India: legacy fonts, token fertility, cross-script retrieval — Sarvam Vision + BGE-M3 + ChrF evaluation. Evaluate with RAGAS: faithfulness, context precision/recall, answer relevancy.
